# SIT744 Assignment 4 — Student Submission Template

This template is for organising your submission and reporting results consistently. It is **not** a worked solution. Replace all bracketed text and add your own implementation code where indicated.


## Student and submission details

- **Student name:** Keoma Borges Martins Ferreira Reis
- **Student ID:** 224790234
- **Unit code:** SIT744
- **Assignment:** Assignment 4
- **Date:** 18/05/2026


## Notes

- Keep answers short and to the point.
- Keep code outputs clean; remove excessive logs before exporting to PDF.
- Use the fixed assignment protocol from the brief: `distilgpt2`, the same tokenizer, the required three text files, perplexity, and the fixed generation prompt.
- Use one coherent pipeline across Tasks 1--3.
- The purpose of this template is to make reporting consistent for marking. You are responsible for writing and explaining your own code.


## Fixed protocol summary

| Item | Required setting |
|---|---|
| Base model | `distilgpt2` |
| Tokenizer | tokenizer associated with `distilgpt2` |
| Training text | `undrip_train.txt` |
| Validation text | `undrip_val.txt` |
| Held-out related evaluation text | `economic_test.txt` |
| Required metric | perplexity |
| Generation prompt | `Economic self-determination for First Nations peoples requires` |
| Full fine-tuning | use the fixed configuration from the assignment brief/starter guidance |
| LoRA | use the fixed LoRA configuration from the assignment brief/starter guidance |


## Fixed configuration values for required results

Use the following configuration values for the required Task 2 and Task 3 comparison. These cells specify the controlled protocol only; they do **not** provide the implementation. Do not change these values for the main results reported in the master results table.


In [34]:
# Fixed configuration values for Assignment 4.
# These are provided so all students report results under the same protocol.

BASE_MODEL = "distilgpt2"

TEXT_FILES = {
    "train": "undrip_train.txt",
    "validation": "undrip_val.txt",
    "test": "economic_test.txt",
}

GENERATION_PROMPT = "Economic self-determination for First Nations peoples requires"

TOKENISATION_CONFIG = {
    "block_size": 128,
}

# Settings for perplexity evaluation.
# These are not Hugging Face TrainingArguments.
PERPLEXITY_MAX_LENGTH = 128
PERPLEXITY_STRIDE = 128

GENERATION_CONFIG = {
    "max_new_tokens": 80,
    "do_sample": False,
}


FULL_FINE_TUNING_CONFIG = {
    "output_dir": "./distilgpt2-full-finetuned",
    "num_train_epochs": 1,
    "learning_rate": 5e-5,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "save_strategy": "no",
    "report_to": "none",
}

LORA_CONFIG = {
    "r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "target_modules": ["c_attn"],
    "bias": "none",
    "task_type": "CAUSAL_LM",
}

# LORA_CONFIG controls the LoRA adapter itself: rank, scaling, dropout,
# and target modules.

# LORA_TRAINING_CONFIG controls how the LoRA adapter is trained. 
LORA_TRAINING_CONFIG = {
    "output_dir": "./distilgpt2-lora-finetuned",
    "num_train_epochs": 1,
    "learning_rate": 1e-4,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "save_strategy": "no",
    "report_to": "none",
}


In [35]:
# Optional setup for Colab or a fresh environment.
# Uncomment and run this cell if the required packages are not installed.
# !pip install transformers datasets accelerate peft


## Required tokenizer and model setup

In [36]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# GPT-style tokenizers such as distilgpt2 do not define a separate padding token by default.
# Use the EOS token as the padding token so that batched training/evaluation can run.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

pretrained_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
pretrained_model.resize_token_embeddings(len(tokenizer))
pretrained_model.config.pad_token_id = tokenizer.pad_token_id

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 13265.38it/s]


## Environment and imports

In [37]:
# Add or modify imports as needed for your implementation.
# Keep this cell concise.

import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import math
import torch

from pathlib import Path


## Reproducibility

In [38]:
# Reproducibility
# This reduces run-to-run variation in data shuffling, model initialisation, and generation.
# Exact reproducibility is still not guaranteed on all GPU/CUDA setups.
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

print(f"Seed set to {SEED}")


Seed set to 42


# Task 1 Data preparation and pretrained baseline (25%)

In this task, prepare the three required text files and establish the pretrained baseline.


## Task 1.1 Source documents and extraction method

**Fine-tuning source document:** [Inquiry into the application of the United Nations Declaration on the Rights of Indigenous Peoples in Australia](https://www.aph.gov.au/Parliamentary_Business/Committees/Joint/Aboriginal_and_Torres_Strait_Islander_Affairs/UNDRIP/Report)

**Held-out related evaluation source document:** [Inquiry into economic self-determination and opportunities for First Nations Australians](https://www.aph.gov.au/Parliamentary_Business/Committees/Joint/Aboriginal_and_Torres_Strait_Islander_Affairs/Economicself-determinati/Report)

**Extraction method**  
I downloaded the 2 public reports and converted the PDF files to plain text using pdf24.org. I saved the extracted text as `undrip_raw.txt` and `economic_raw.txt`. I then used the provided preprocessing scaffold to clean the texts and split into an 80/20 train/validation split, and save the cleaned economic report as the held-out test file.


In [39]:
def clean_text(text):
    lines = text.splitlines()
    lines = [line.strip() for line in lines]
    lines = [line for line in lines if len(line) > 0]
    return "\n".join(lines)

def split_text(text, train_ratio=0.8):
    split_index = int(len(text) * train_ratio)
    return text[:split_index], text[split_index:]

if __name__ == '__main__':
    # Place your extracted raw plain text into these two files first.
    undrip_raw = Path("undrip_raw.txt").read_text(encoding="utf-8")
    economic_raw = Path("economic_raw.txt").read_text(encoding="utf-8")

    undrip_clean = clean_text(undrip_raw)
    economic_clean = clean_text(economic_raw)

    undrip_train, undrip_val = split_text(undrip_clean, train_ratio=0.8)

    Path("undrip_train.txt").write_text(undrip_train, encoding="utf-8")
    Path("undrip_val.txt").write_text(undrip_val, encoding="utf-8")
    Path("economic_test.txt").write_text(economic_clean, encoding="utf-8")


## Task 1.2 Data-preparation table

| File | Role | Character count | Approximate token count |
|---|---|----------------:|------------------------:|
| `undrip_train.txt` | training text |          330859 |                   66352 |
| `undrip_val.txt` | validation text |           82715 |                   16276 |
| `economic_test.txt` | held-out related evaluation text |          452566 |                   99776 |


In [40]:
# Optional: code used to compute character counts and approximate token counts.
def count(file_path):
    text = Path(file_path).read_text(encoding="utf-8")
    token_ids = tokenizer.encode(text, add_special_tokens=False)

    return {
        "char count": len(text),
        "token count": len(token_ids),
    }

print(count("undrip_train.txt"))
print(count("undrip_val.txt"))
print(count("economic_test.txt"))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (66352 > 1024). Running this sequence through the model will result in indexing errors


{'char count': 330859, 'token count': 66352}
{'char count': 82715, 'token count': 16276}
{'char count': 452566, 'token count': 99776}


## Task 1.3 Pretrained baseline evaluation

Evaluate pretrained `distilgpt2` on `undrip_val.txt` and `economic_test.txt`. Complete only the pretrained row of the master results table at this stage.


In [41]:
# Pretrained baseline loading and perplexity evaluation code goes here.
# Use pretrained_model for the pretrained baseline.

# Perplexity guidance:
# Use model.eval(), disable gradients, and use labels=input_ids for causal language modelling.
# For long texts, use PERPLEXITY_MAX_LENGTH and PERPLEXITY_STRIDE.
def perplexity(model, tokenizer, file_path, max_length=PERPLEXITY_MAX_LENGTH, stride=PERPLEXITY_STRIDE):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    text = Path(file_path).read_text(encoding="utf-8")
    encodings = tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids.to(device)

    seq_len = input_ids.size(1)
    nll_sum = 0.0
    n_tokens = 0
    prev_end_loc = 0

    with torch.no_grad():
        for begin_loc in range(0, seq_len, stride):
            end_loc = min(begin_loc + max_length, seq_len)
            trg_len = end_loc - prev_end_loc

            input_ids_slice = input_ids[:, begin_loc:end_loc]
            target_ids = input_ids_slice.clone()
            target_ids[:, :-trg_len] = -100

            outputs = model(input_ids_slice, labels=target_ids)
            neg_log_likelihood = outputs.loss

            nll_sum += neg_log_likelihood.item() * trg_len
            n_tokens += trg_len

            prev_end_loc = end_loc

            if end_loc == seq_len:
                break

    return math.exp(nll_sum / n_tokens)

pretrained_val_ppl = perplexity(
    pretrained_model,
    tokenizer,
    "undrip_val.txt",
)

pretrained_economic_ppl = perplexity(
    pretrained_model,
    tokenizer,
    "economic_test.txt",
)

print(f"pretrained_val_ppl: {pretrained_val_ppl}")
print(f"pretrained_economic_ppl: {pretrained_economic_ppl}")

pretrained_val_ppl: 53.34027397216509
pretrained_economic_ppl: 60.87435530123492


## Master results table

Complete this table across Tasks 1-3.

Training time depends on the runtime environment, so use it mainly for comparing your own full fine-tuning and LoRA runs under the same setup.

| Model state | Parameters updated during adaptation | Training time (minutes; compare within the same runtime) | PPL on UNDRIP validation | PPL on Economic test |
|---|---:|---:|---:|---------------------:|
| Pretrained `distilgpt2` | 0  | 0 | 53.340 |               60.874 |
| Full fine-tuned `distilgpt2` | [Replace] | [Replace] | [Replace] |            [Replace] |
| LoRA-adapted `distilgpt2` | [Replace] | [Replace] | [Replace] |            [Replace] |


# Task 2 Full fine-tuning (25%)

Fine-tune `distilgpt2` once using the fixed full fine-tuning configuration. Report the result in the master results table.


In [42]:
# Start full fine-tuning from a fresh pretrained model.
full_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
full_model.resize_token_embeddings(len(tokenizer))
full_model.config.pad_token_id = tokenizer.pad_token_id

# Full fine-tuning code goes here.
# Use FULL_FINE_TUNING_CONFIG for the required main result.
# Keep the base model, tokenizer, training text, validation text, test text,
# evaluation metric, and generation prompt fixed.


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 15850.39it/s]


## Task 2 evidence

Include either a training-loss curve or a compact final training-loss record.


In [43]:
# Plot the training-loss curve or print a compact final training-loss record here.


# Task 3 LoRA adaptation and comparison (25%)

Train a LoRA-adapted version of `distilgpt2` using the fixed LoRA configuration. Reload a fresh distilgpt2 base model before applying LoRA. Do not apply LoRA to the full fine-tuned model from Task 2.
Report the result in the master results table.


In [44]:
# Start LoRA from a fresh pretrained model, not from the full fine-tuned model.
lora_base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
lora_base_model.resize_token_embeddings(len(tokenizer))
lora_base_model.config.pad_token_id = tokenizer.pad_token_id

# LoRA adaptation and evaluation code goes here.
# Use LORA_CONFIG for the required main result.
lora_training_args = TrainingArguments(**LORA_TRAINING_CONFIG)
# Apply LoRA to lora_base_model and store the adapted model as lora_model.
# Do not apply LoRA on top of the full fine-tuned model for the required comparison.


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 11183.24it/s]


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`

## Generation comparison table

Use the fixed generation prompt exactly. Keep outputs short enough to be readable in the exported PDF.

| Prompt | Pretrained output | Full fine-tuned output | LoRA-adapted output | One-sentence comparison |
|---|---|---|---|---|
| `Economic self-determination for First Nations peoples requires` | [Replace] | [Replace] | [Replace] | [Replace] |


In [ ]:
# Generation code for pretrained_model, full_model, and lora_model goes here.
# Use GENERATION_PROMPT and GENERATION_CONFIG for the required comparison.


# Task 4 Interpretation and learning reflection (20%)

This task is about explanation and evidence of your individual learning.


## Task 4A Technical interpretation (15%)

Answer the following questions in at most **300 words** total. Your answers must refer to your own master results table and generation comparison table.

1. Using your reported perplexity values, what do your results show about adaptation to `undrip_val.txt` versus transfer to `economic_test.txt`?
2. Using your reported trainable-parameter count, training time, perplexity values, and generation comparison, what trade-off do you observe between full fine-tuning and LoRA?
3. Based on your own results, which method would you recommend under limited compute for a quick domain-adaptation baseline, and why?

Before writing, look at your master results table and generation comparison table. Your discussion should refer to your own reported values. For example: "Full fine-tuning reduced PPL on UNDRIP validation from X to Y, while LoRA reduced it to Z."

**Your response:**  
Write your Task 4A response here. Keep the total response to at most 300 words and base your claims on your reported results.


## Task 4B Learning reflection (5%)

Answer the following question in at most **150 words**.

Over the 11 weeks of this unit, what is one intuition about deep learning that has changed, become clearer, or become more nuanced for you?

Your answer should include:

- one earlier intuition, assumption, or misconception;
- the concept, activity, result, lecture, practical, or example that changed or refined it;
- what you now understand differently;
- how this would affect the way you approach a future deep learning problem.

**Your response:**  
[Replace]


# Final checklist

- [ ] The notebook filename follows the required format.
- [ ] The PDF export filename follows the required format.
- [ ] `undrip_train.txt`, `undrip_val.txt`, and `economic_test.txt` are submitted.
- [ ] The 80/20 UNDRIP split is followed.
- [ ] All modelling tasks use the text files created in Task 1.
- [ ] The data-preparation table is completed.
- [ ] The master results table is completed.
- [ ] A training-loss curve or compact final training-loss record is included.
- [ ] The generation comparison table is completed.
- [ ] The fixed generation prompt is used exactly.
- [ ] Perplexity is reported consistently.
- [ ] The technical interpretation is concise.
- [ ] My technical interpretation refers to my own master results table and generation comparison table.
- [ ] The learning reflection identifies a concrete change in understanding.
- [ ] Code outputs are cleaned before PDF export.
- [ ] Cloud resources are shut down after use.
